# RAG-Trained Weights — Plain-Text Inference (No Retrieval)

This notebook evaluates the RAG-trained classifiers in `weigths/weights_rac_best_hyperparameters/` **without any retrieval at inference time**.
The models were fine-tuned on augmented inputs (`query [SEP] neighbor1 [SEP] ...`) but here receive only the raw query text.

**Purpose:** isolate the contribution of training-time augmentation from inference-time retrieval.
If scores are close to the full RAG pipeline, the retrieval at inference time adds little value.
If scores drop, the model relies on neighbors to make good predictions.

**Configurable dimensions (edit Cell 2):**
| Variable | Options |
|---|---|
| `SELECTED_MODELS` | `bert`, `hatebert`, `roberta` |
| `SELECTED_INDEX_TYPES` | `training`, `documents`, `full` |
| `SELECTED_DATASETS` | `IHC`, `ISHate`, `Vicomtech` |

## 1. Imports

In [ ]:
import os
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from datasets import load_dataset, Dataset
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path('..').resolve()))

## 2. Configuration

Edit `SELECTED_MODELS`, `SELECTED_INDEX_TYPES`, and `SELECTED_DATASETS` to choose what to evaluate.

In [ ]:
ROOT_DIR        = Path('../..')
WEIGHTS_RAC_DIR = ROOT_DIR / 'weigths' / 'weights_rac_best_hyperparameters'

MAX_LENGTH = 256
BATCH_SIZE = 32

# === What to evaluate — edit these lists ===
SELECTED_MODELS      = ['bert', 'roberta']             # 'bert' | 'hatebert' | 'roberta'
SELECTED_INDEX_TYPES = ['example', 'knowledge', 'full']
SELECTED_DATASETS    = ['IHC', 'ISHate', 'Vicomtech']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device           : {device}')
print(f'Models           : {SELECTED_MODELS}')
print(f'Index types      : {SELECTED_INDEX_TYPES}')
print(f'Datasets         : {SELECTED_DATASETS}')

## 3. Load Datasets

Only datasets in `SELECTED_DATASETS` are loaded.

In [ ]:
from data_loaders import load_ihc_implicit_only, load_ishate_binary, load_vicomtech

DATASETS = {}
if 'IHC' in SELECTED_DATASETS:
    test_ihc = load_ihc_implicit_only(seed=42)
    DATASETS['IHC'] = {'test': test_ihc, 'text_col': 'post'}
    print(f'IHC       — test: {len(test_ihc):,}')

if 'ISHate' in SELECTED_DATASETS:
    _, test_ishate = load_ishate_binary()
    DATASETS['ISHate'] = {'test': test_ishate, 'text_col': 'text'}
    print(f'ISHate    — test: {len(test_ishate):,}')

if 'Vicomtech' in SELECTED_DATASETS:
    test_vicomtech = load_vicomtech(split='test')
    DATASETS['Vicomtech'] = {'test': test_vicomtech, 'text_col': 'text'}
    print(f'Vicomtech — test: {len(test_vicomtech):,}')

## 4. Model Registry

Builds all `(model, index_type, dataset)` combinations and checks which weights exist on disk.

In [ ]:
MODELS_CONFIG = [
    {
        'path':       WEIGHTS_RAC_DIR / model / 'sbert' / index_type / dataset,
        'label':      f'{model} / sbert / {index_type} / {dataset}',
        'model_name': model,
        'index_type': index_type,
        'dataset':    dataset,
        'text_col':   DATASETS[dataset]['text_col'],
    }
    for model in SELECTED_MODELS
    for index_type in SELECTED_INDEX_TYPES
    for dataset in SELECTED_DATASETS
]

print(f"{'Model / Index / Dataset':<45} Weights?")
print('-' * 55)
for m in MODELS_CONFIG:
    has = (m['path'] / 'model.safetensors').exists() or (m['path'] / 'pytorch_model.bin').exists()
    print(f"{m['label']:<45} {'✓' if has else '✗  (missing)'}")

## 5. Helpers

In [ ]:
from training_utils import compute_metrics, tokenize_plain

## 6. Evaluation Loop

For each available model: tokenize plain text (no retrieval) → predict → store metrics.

In [ ]:
results = {}

eval_args = TrainingArguments(
    output_dir='./tmp_eval',
    per_device_eval_batch_size=BATCH_SIZE,
    report_to='none',
)

for entry in MODELS_CONFIG:
    has_weights = (entry['path'] / 'model.safetensors').exists() or (entry['path'] / 'pytorch_model.bin').exists()
    if not has_weights:
        print(f"[skip] {entry['label']} — no weights on disk")
        continue

    print(f"\n{'='*60}")
    print(f"{entry['label']}")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(entry['path'])
    test_ds   = DATASETS[entry['dataset']]['test']
    tok_test  = tokenize_plain(test_ds, tokenizer, entry['text_col'])

    model   = AutoModelForSequenceClassification.from_pretrained(entry['path'])
    trainer = Trainer(model=model, args=eval_args, compute_metrics=compute_metrics)

    preds_out = trainer.predict(tok_test)
    preds     = np.argmax(preds_out.predictions, axis=-1)
    labels    = list(test_ds['label'])

    print(classification_report(labels, preds, target_names=['Non-HS', 'HS']))

    results[entry['label']] = {
        'model':      entry['model_name'],
        'index_type': entry['index_type'],
        'dataset':    entry['dataset'],
        'macro_f1':   f1_score(labels, preds, average='macro',  zero_division=0),
        'macro_p':    precision_score(labels, preds, average='macro', zero_division=0),
        'macro_r':    recall_score(labels, preds, average='macro',    zero_division=0),
    }

    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()

## 7. Results — One Table per Dataset

In [ ]:
for ds_name in SELECTED_DATASETS:
    ds_results = {
        k: v for k, v in results.items() if v['dataset'] == ds_name
    }
    if not ds_results:
        print(f'No results for {ds_name}\n')
        continue

    rows = {}
    for label, vals in ds_results.items():
        row_key = f"{vals['model']} / sbert / {vals['index_type']}"
        rows[row_key] = {
            'Macro F1':        vals['macro_f1'],
            'Macro Precision': vals['macro_p'],
            'Macro Recall':    vals['macro_r'],
        }

    df = pd.DataFrame(rows).T
    df.index.name = 'Model / Index'

    display(
        df.style
        .format('{:.3f}')
        .highlight_max(axis=0, props='font-weight: bold; background-color: #d4f1d4')
        .set_caption(f'{ds_name} — RAG weights, plain-text inference (no retrieval)')
    )